In [ ]:
# Group Members: Artificial Intelligence.....
# 1. 105241- William Kagoiyo Wabuiya
# 2. 170459- Ryan Maxine.
# 3. 169258- Samuel K. Ptolomy.
# 4. 133865- Franklyne Olunga Mayende.
# 5. 088251 - Claire Naibo

# "Dataset: Stars Classification"
#------------------------------------------------------
#Eager Classifier: SVM- Support Vector Machine"
#------------------------------------------------------

In [1]:
# QN 1: Importing the necessary libraries and loading the dataset
# We start by importing every library we will need throughout
# this notebook. This includes libraries for data manipulation, visualization, and machine learning.

import pandas as pd          # for loading and manipulating our data table
import numpy as np           # for numerical operations
import matplotlib.pyplot as plt  # for drawing charts and graphs
import seaborn as sns        # for prettier, more informative charts

# Machine learning tools from scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC                          # Support Vector Classifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score
)

import warnings
warnings.filterwarnings('ignore')   # keeps our output clean during presentation

# ---------------------------------------------------------------
# Loading the dataset.
# ---------------------------------------------------------------
df = pd.read_csv('Stars.csv')

print("Dataset loaded successfully!")
print(f"Shape of dataset: {df.shape}")   # rows x columns
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 5 rows of the raw data:")
df.head()


Dataset loaded successfully!
Shape of dataset: (240, 8)

Column names:
['Temperature (K)', 'Luminosity (L/Lo)', 'Radius (R/Ro)', 'Absolute magnitude (Mv)', 'Star type', 'Star category', 'Star color', 'Spectral Class']

First 5 rows of the raw data:


,Temperature (K),Luminosity (L/Lo),Radius (R/Ro),Absolute magnitude (Mv),Star type,Star category,Star color,Spectral Class
0,3068,0.002400,0.1700,16.12,0,Brown Dwarf,Red,M
1,3042,0.000500,0.1542,16.60,0,Brown Dwarf,Red,M
2,2600,0.000300,0.1020,18.70,0,Brown Dwarf,Red,M
3,2800,0.000200,0.1600,16.65,0,Brown Dwarf,Red,M
4,1939,0.000138,0.1030,20.06,0,Brown Dwarf,Red,M


In [3]:
# =============================================================
# QN 2 (a) : Data Cleaning + Exploratory Data Analysis (EDA)
# =============================================================
# Before we build any model, we need to understand our data
# and make sure it is clean. "Garbage in, garbage out" is a
# real risk in machine learning — a dirty dataset will produce
# an unreliable model no matter how sophisticated the algorithm.

# ---------------------------------------------------------------
# STEP 1: Basic overview — what does the data look like?
# ---------------------------------------------------------------
print("=" * 60)
print("STEP 1: Basic Data Overview")
print("=" * 60)
print(df.info())           # data types and non-null counts per column
print("\nDescriptive Statistics (numerical columns):")
print(df.describe())       # min, max, mean, std for each numeric column

# ---------------------------------------------------------------
# STEP 2: Check for missing values
# A missing value means a cell in our table has no data.
# If we leave missing values in, they will cause errors when
# we try to train models later.
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 2: Missing Values Check")
print("=" * 60)
print(df.isnull().sum())

# ---------------------------------------------------------------
# STEP 3: Check for duplicate rows
# A duplicate row is a row that is an exact copy of another.
# Keeping duplicates can bias our model by making it learn
# the same example more than once.
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 3: Duplicate Rows Check")
print("=" * 60)
print(f"Number of duplicate rows found: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df.shape}")

# ---------------------------------------------------------------
# STEP 4: Check class balance in the target column
# Our target is 'Star type' — a number from 0 to 5.
# We check if all 6 classes have roughly equal representation.
# Severe imbalance can make a classifier biased toward the
# majority class.
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("STEP 4: Target Class Distribution (Star type)")
print("=" * 60)
print(df['Star type'].value_counts())

# ---------------------------------------------------------------
# STEP 5: Map the numeric Star type to its readable label
# so our plots are self-explanatory to the lecturer.
# ---------------------------------------------------------------
star_labels = {
    0: 'Brown Dwarf',
    1: 'Red Dwarf',
    2: 'White Dwarf',
    3: 'Main Sequence',
    4: 'Supergiant',
    5: 'Hypergiant'
}
df['Star category'] = df['Star type'].map(star_labels)

print("\nCleaned dataset — first 5 rows:")
df.head()

STEP 1: Basic Data Overview
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Temperature (K)          240 non-null    int64  
 1   Luminosity (L/Lo)        240 non-null    float64
 2   Radius (R/Ro)            240 non-null    float64
 3   Absolute magnitude (Mv)  240 non-null    float64
 4   Star type                240 non-null    int64  
 5   Star category            240 non-null    str    
 6   Star color               240 non-null    str    
 7   Spectral Class           240 non-null    str    
dtypes: float64(3), int64(2), str(3)
memory usage: 15.1 KB
None

Descriptive Statistics (numerical columns):
       Temperature (K)  Luminosity (L/Lo)  Radius (R/Ro)  \
count       240.000000         240.000000     240.000000   
mean      10497.462500      107188.361635     237.157781   
std        9552.425037      179432.244940     517.

,Temperature (K),Luminosity (L/Lo),Radius (R/Ro),Absolute magnitude (Mv),Star type,Star category,Star color,Spectral Class
0,3068,0.002400,0.1700,16.12,0,Brown Dwarf,Red,M
1,3042,0.000500,0.1542,16.60,0,Brown Dwarf,Red,M
2,2600,0.000300,0.1020,18.70,0,Brown Dwarf,Red,M
3,2800,0.000200,0.1600,16.65,0,Brown Dwarf,Red,M
4,1939,0.000138,0.1030,20.06,0,Brown Dwarf,Red,M
